In [1]:
import os
import copy
import random
import time
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
import networkx as nx
from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv
import community.community_louvain as community_louvain
from torch_geometric.data import Data
from pathlib import Path

/home/kmc/work/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# Graph utilities
# -------------------------
def generate_graph(file):
    return nx.read_edgelist(file, nodetype=int)
def build_adj_structures(G):
    nodes_sorted = sorted(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes_sorted)}
    adjacency = {n: set(G.neighbors(n)) | {n} for n in nodes_sorted}  # include self-loop
    deg = {n: len(adjacency[n]) for n in nodes_sorted}
    return adjacency, deg, nodes_sorted, node_to_idx

In [3]:
# incremental_gcn_with_AX_cache.py
# Updated script: incremental GCN using cached AX (A_hat X) for the first-layer propagation.
# Key idea: Precompute A_hat @ X once (or update rows locally after graph perturbations)
# and feed it to a linear layer instead of running the full GCNConv for the first hop.



# -------------------------
# Utilities for adjacency normalization and AX caching
# -------------------------
# -------------------------
# NEW: Build k-hop induced subgraph around changed region
# -------------------------

def build_k_hop_subgraph(G, affected_nodes, k=2):
    """
    Return the induced subgraph of nodes within k hops
    of any node in affected_nodes.
    """
    frontier = set(affected_nodes)
    visited = set(frontier)

    for _ in range(k):
        new = set()
        for u in frontier:
            new.update(G.neighbors(u))
        new -= visited
        visited |= new
        frontier = new

    return G.subgraph(sorted(visited)).copy(), visited


def build_adj_structures(G):
    """Return adjacency lists (including self-loop) and degrees (with self-loop).
    adjacency: dict node -> set(neighbors incl self)
    deg: dict node -> degree (including self-loop)
    nodes_sorted: list of nodes in deterministic order
    node_to_idx: mapping node -> index
    """
    nodes_sorted = sorted(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes_sorted)}

    adjacency = {n: set(G.neighbors(n)) for n in nodes_sorted}
    # include self-loop
    for n in nodes_sorted:
        adjacency[n].add(n)

    deg = {n: len(adjacency[n]) for n in nodes_sorted}

    return adjacency, deg, nodes_sorted, node_to_idx


def compute_AX_full(adjacency, deg, nodes_sorted, node_to_idx, X):
    """Compute A_hat X where A_hat = D^{-1/2} A D^{-1/2} (A includes self-loops).
    X is a numpy or torch array of shape (N, F). Returns torch tensor (N, F).
    """
    # Make sure X is a torch tensor on cpu
    if isinstance(X, np.ndarray):
        X_t = torch.tensor(X, dtype=torch.float)
    else:
        X_t = X.clone().detach().cpu()

    N, F = X_t.shape
    AX = torch.zeros_like(X_t)

    # row-wise computation: for node i
    for i, n in enumerate(nodes_sorted):
        deg_i = deg[n]
        inv_sqrt_deg_i = 1.0 / np.sqrt(deg_i)
        nbrs = adjacency[n]
        acc = torch.zeros(F)
        for m in nbrs:
            j = node_to_idx[m]
            deg_j = deg[m]
            inv_sqrt_deg_j = 1.0 / np.sqrt(deg_j)
            weight = inv_sqrt_deg_i * inv_sqrt_deg_j
            acc += weight * X_t[j]
        AX[i] = acc
    return AX


def update_AX_rows(adjacency, deg, nodes_sorted, node_to_idx, X, AX, rows_to_update):
    """Recompute AX rows for nodes in rows_to_update, including neighbors whose
    degree-normalization changed. This now performs the mathematically correct
    update for A_hat = D^{-1/2} A D^{-1/2}.
    """
    # ensure tensors
    if isinstance(X, np.ndarray):
        X_t = torch.tensor(X, dtype=torch.float)
    else:
        X_t = X.clone().cpu()

    AX_new = AX.clone()

    # compute new degrees after perturbation
    for n in adjacency:
        deg[n] = len(adjacency[n])

    # compute 1/sqrt(deg)
    inv_sqrt_deg = {n: 1.0 / np.sqrt(max(deg[n], 1e-12)) for n in adjacency}

    # expand rows_to_update to include normalization dependents
    expanded = set(rows_to_update)
    for n in rows_to_update:
        for nbr in adjacency[n]:
            expanded.add(nbr)

    # rebuild AX rows
    for n in expanded:
        i = node_to_idx[n]
        acc = torch.zeros_like(AX_new[i])
        inv_i = inv_sqrt_deg[n]
        for m in adjacency[n]:
            j = node_to_idx[m]
            acc += inv_i * inv_sqrt_deg[m] * X_t[j]
        AX_new[i] = acc
    return AX_new


def normalize_by_min_node(partition):
    comm_to_nodes = defaultdict(list)
    for node, cid in partition.items():
        comm_to_nodes[cid].append(node)
    sorted_comms = sorted(comm_to_nodes.items(), key=lambda x: min(x[1]))
    new_partition = {}
    for new_label, (old_cid, nodes) in enumerate(sorted_comms):
        for node in nodes:
            new_partition[node] = new_label
    return new_partition


def generate_labels(G):
    partition_raw = community_louvain.best_partition(G)
    partition = normalize_by_min_node(partition_raw)
    print("Community counts:", Counter(partition.values()))
    return partition


def generate_data(G, node_labels):
    # attach y
    for n in G.nodes():
        G.nodes[n]['y'] = node_labels[n]

    # features: degree, clustering, pagerank (same as before)
    nodes_sorted = sorted(G.nodes())
    degrees = np.array([G.degree(n) for n in nodes_sorted], dtype=float)
    clustering = np.array([nx.clustering(G, n) for n in nodes_sorted], dtype=float)
    pagerank = np.array(list(nx.pagerank(G, alpha=0.85).values()), dtype=float)
    X = np.vstack([degrees, clustering, pagerank]).T
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-9)

    # map back into G nodes in sorted order
    for i, n in enumerate(nodes_sorted):
        G.nodes[n]['x'] = X[i]

    data = from_networkx(G)
    data.x = torch.tensor(np.vstack([G.nodes[n]['x'] for n in nodes_sorted]), dtype=torch.float)
    data.y = torch.tensor([G.nodes[n]['y'] for n in nodes_sorted], dtype=torch.long)

    # store node ordering to maintain consistent indexing
    data.nodes_sorted = nodes_sorted
    data.node_to_idx = {n: i for i, n in enumerate(nodes_sorted)}
    return data

# -------------------------
# Model: first-hop uses cached AX
# -------------------------
class IncrementalGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # first "propagation" replaced by a Linear applied to AX (precomputed).
        self.lin1 = torch.nn.Linear(in_channels, hidden_channels, bias=True)
        # keep second GCNConv for mixing (on hidden representations)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        # data.ax should be the precomputed A_hat @ X aligned with data.nodes_sorted
        ax = data.ax.to(data.x.device)
        x = self.lin1(ax)
        x = F.leaky_relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        # Now perform the second hop using conv2 (it expects node features aligned to edge_index)
        x = self.conv2(x, data.edge_index)
        return F.log_softmax(x, dim=1)

# -------------------------
# Perturbation: slightly modified to also update adjacency structures cleanly
# -------------------------

def perturb_graph_to_change_communities(
        graph,
        node_labels,
        num_add=50,
        num_remove=50,
        affected_ratio=(0.05, 0.10),
        seed=None
    ):
    if seed is not None:
        random.seed(seed)

    nodes = list(graph.nodes())
    N = len(nodes)
    min_k = int(affected_ratio[0] * N)
    max_k = int(affected_ratio[1] * N)
    k = random.randint(min_k, max_k)
    affected_nodes = set(random.sample(nodes, k))

    # simple removals and additions similar to your script
    possible_removals = []
    for u in affected_nodes:
        u_comm = node_labels[u]
        for v in list(graph.neighbors(u)):
            if node_labels[v] == u_comm or random.random() > 0.65:
                possible_removals.append((u, v))
    random.shuffle(possible_removals)
    edges_to_remove = possible_removals[:num_remove]
    removed_edges = []
    for u, v in edges_to_remove:
        if graph.has_edge(u, v):
            graph.remove_edge(u, v)
            removed_edges.append((u, v))

    added_edges = []
    comm_to_nodes = defaultdict(list)
    for n, c in node_labels.items():
        comm_to_nodes[c].append(n)

    attempts = 0
    max_attempts = num_add * 20
    while len(added_edges) < num_add and attempts < max_attempts:
        u = random.choice(list(affected_nodes))
        u_comm = node_labels[u]
        other_comms = [c for c in comm_to_nodes.keys() if c != u_comm]
        if not other_comms:
            break
        if random.random() > 0.4:
            tgt_comm = random.choice(other_comms)
            inter = list(set(comm_to_nodes[tgt_comm]).intersection(affected_nodes))
            if not inter:
                attempts += 1
                continue
            v = random.choice(inter)
            if u != v and not graph.has_edge(u, v):
                graph.add_edge(u, v)
                added_edges.append((u, v))
        else:
            v = random.choice(comm_to_nodes[u_comm])
            if u != v and not graph.has_edge(u, v):
                graph.add_edge(u, v)
                added_edges.append((u, v))
        attempts += 1

    return graph, added_edges, removed_edges, affected_nodes

def generate_G_update(G,node_labels,num_nodes=100):
    print(G)
    count=0
    c=0
    added_edges = []
    removed_edges = []
    affected_nodes = set()
    for i in G:
        comm = node_labels[i]
        ncomm = random.choice(list({0,1,2,3}.difference(set([comm]))))
        # print(comm,ncomm)
        cc=0
        for j in G:
            if node_labels[j]==ncomm:
                if G.has_edge(i,j)==False:
                    G.add_edge(i,j)
                    added_edges.append((i,j))
                    affected_nodes.add(i)
                    affected_nodes.add(j)
                    cc+=1
            if cc==500:
                break
        count+=500
        c+=1
        if c==num_nodes:
            break
    # print(count)
    
    print(G)
    # 2. Run Louvain
    new_raw = compute_communities(G)
    
    # 3. Stabilize labels (very important)
    new_labels, mapping = stabilize_partition(node_labels, new_raw)
    return G, added_edges, removed_edges, affected_nodes, new_labels
# -----------------------------------
# Utility: Convert node->cid to cid->nodes
# -----------------------------------
def group_to_nodes(partition):
    d = defaultdict(list)
    for n,c in partition.items():
        d[c].append(n)
    return d

# -----------------------------------
# Utility: Compute Louvain communities
# -----------------------------------
import community.community_louvain as community_louvain
class PureGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, data.edge_index)
        return F.log_softmax(x, dim=1)
def compute_communities(G, seed=42):
    return community_louvain.best_partition(G, random_state=seed)

# -----------------------------------
# Core: Stabilize community labels over time
# -----------------------------------
def stabilize_partition(old_part, new_part):
    old_groups = group_to_nodes(old_part)
    new_groups = group_to_nodes(new_part)

    # Compute overlaps: old_c -> new_c -> |intersection|
    overlaps = {}
    for old_c, old_nodes in old_groups.items():
        old_set = set(old_nodes)
        row = {}
        for new_c, new_nodes in new_groups.items():
            row[new_c] = len(old_set.intersection(new_nodes))
        overlaps[old_c] = row

    # Greedy matching of (old_c, new_c)
    assigned_old = set()
    assigned_new = set()
    mapping = {}

    pairs = []
    for old_c, row in overlaps.items():
        for new_c, ov in row.items():
            pairs.append((ov, old_c, new_c))

    pairs.sort(reverse=True)  # largest overlap first

    for ov, old_c, new_c in pairs:
        if ov == 0:
            break
        if old_c not in assigned_old and new_c not in assigned_new:
            mapping[new_c] = old_c
            assigned_old.add(old_c)
            assigned_new.add(new_c)

    # Create new labels for unmapped new communities
    next_label = max(old_groups.keys()) + 1 if old_groups else 0
    for new_c in new_groups:
        if new_c not in mapping:
            mapping[new_c] = next_label
            next_label += 1

    # Build stabilized new partition
    stable_new = {node: mapping[new_part[node]] for node in new_part}

    return stable_new, mapping

# -----------------------------------
# Detect which nodes changed communities
# -----------------------------------
def find_changed_nodes(old_labels, new_labels):
    return [n for n in old_labels if old_labels[n] != new_labels[n]]


# -------------------------
# Train / evaluate utilities adapted to incremental model
# -------------------------



def prepare_ax_and_structures(G, data):
    adjacency, deg, nodes_sorted, node_to_idx = build_adj_structures(G)
    # data.x is aligned with nodes_sorted (ensured by generate_data)
    X = data.x.clone().cpu()
    AX = compute_AX_full(adjacency, deg, nodes_sorted, node_to_idx, X)
    # store
    data.adjacency = adjacency
    data.deg = deg
    data.nodes_sorted = nodes_sorted
    data.node_to_idx = node_to_idx
    data.ax = AX
    return data


def train_model(model, data, e=200, lr = 0.01):
    num_nodes = data.num_nodes
    idx = np.arange(num_nodes)
    np.random.shuffle(idx)
    idx = torch.tensor(idx, dtype=torch.long)
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[idx[:int(0.6*num_nodes)]] = True
    val_mask[idx[int(0.6*num_nodes):int(0.8*num_nodes)]] = True
    test_mask[idx[int(0.8*num_nodes):]] = True
    data.train_mask, data.val_mask, data.test_mask = train_mask, val_mask, test_mask

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr, weight_decay=5e-4)
    start = time.time()
    for epoch in range(1, e+1):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            model.eval()
            pred = out.argmax(dim=1)
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean()
            # print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")
    end = time.time()
    print(f"Time taken for training: {end-start:.4f} s")
    return model


def evaluate_model(model, data):
    model.eval()
    pred = model(data).argmax(dim=1)
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean()
    print(f"Test Accuracy: {test_acc:.4f}")
    logits = model(data)
    pred = logits.argmax(dim=1).cpu().numpy()
    true = data.y.cpu().numpy()
    print(classification_report(true[data.test_mask.cpu().numpy()], pred[data.test_mask.cpu().numpy()], digits=4))
    # print(confusion_matrix(true[data.test_mask.cpu().numpy()], pred[data.test_mask.cpu().numpy()]))

# -------------------------
# Fine-tuning after perturbation (use incremental AX update)
# -------------------------

def nodes_for_AX_update(adjacency, changed_nodes):
    """Return set of nodes whose AX rows must be recomputed: changed_nodes + their neighbors."""
    rows = set()
    for n in changed_nodes:
        rows.add(n)
        rows.update(adjacency.get(n, []))
    return rows


def fine_tune_incremental(model, G_updated, data, added_edges, removed_edges, changed_nodes, ep=50, lrr=5e-4):
    # 1) Update adjacency & deg structures in data
    adjacency = data.adjacency
    deg = data.deg
    node_to_idx = data.node_to_idx
    nodes_sorted = data.nodes_sorted

    # apply removals
    for u, v in removed_edges:
        if v in adjacency.get(u, set()):
            adjacency[u].remove(v)
        if u in adjacency.get(v, set()):
            adjacency[v].remove(u)
    # apply additions
    for u, v in added_edges:
        adjacency.setdefault(u, set()).add(v)
        adjacency.setdefault(v, set()).add(u)

    # recompute degrees for impacted nodes
    impacted = set(n for e in (added_edges + removed_edges) for n in e)
    for n in impacted:
        deg[n] = len(adjacency[n])

    # decide which AX rows to recompute
    rows_to_update = nodes_for_AX_update(adjacency, changed_nodes)
    # recompute AX rows incrementally
    data.ax = update_AX_rows(adjacency, deg, nodes_sorted, node_to_idx, data.x.cpu(), data.ax, rows_to_update)

    # create data_sub for fine-tuning: we'll fine-tune on the union of nodes touched in edges and nodes whose labels changed
    # Here we simply fine-tune using full graph but with small epoch count — you can restrict to subgraph if desired.
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lrr, weight_decay=5e-4)
    t_s = time.time()
    for epoch in range(1, ep+1):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            model.eval()
            acc = (out.argmax(dim=1)[data.train_mask] == data.y[data.train_mask]).float().mean()
            # print(f"Fine-tune Epoch {epoch:03d} | Loss: {loss:.4f} | Train-acc: {acc:.4f}")
    t_e = time.time()
    print("finetuning time", t_e-t_s)
    return model

# -------------------------
# NEW: Fine-tuning ONLY on the k-hop subgraph
# -------------------------
def fine_tune_subgraph(model, G_updated, data, affected_nodes, k=2, use_AX=True, epochs=30, lr=5e-4):
    G_sub, sub_nodes = build_k_hop_subgraph(G_updated, affected_nodes, k)
    idx_map = {n:i for i,n in enumerate(sub_nodes)}
    data_sub = from_networkx(G_sub)
    data_sub.x = data.x[[data.node_to_idx[n] for n in sub_nodes]].clone()
    data_sub.y = data.y[[data.node_to_idx[n] for n in sub_nodes]].clone()
    data_sub.nodes_sorted = sub_nodes
    data_sub.node_to_idx = idx_map
    if use_AX:
        adjacency_sub = {n: set(G_sub.neighbors(n)) | {n} for n in sub_nodes}
        deg_sub = {n: len(adjacency_sub[n]) for n in sub_nodes}
        AX_sub = compute_AX_full(adjacency_sub, deg_sub, sub_nodes, idx_map, data_sub.x)
        data_sub.ax = AX_sub
    mask = torch.ones(len(sub_nodes), dtype=torch.bool)
    data_sub.train_mask = mask
    data_sub.val_mask = mask
    data_sub.test_mask = mask
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data_sub = data_sub.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data_sub)
        loss = F.nll_loss(out[mask], data_sub.y[mask])
        loss.backward()
        optimizer.step()
    return model
def fine_tune_incremental_subgraph(
        model,
        G_updated,
        data,
        added_edges,
        removed_edges,
        affected_nodes,
        k=2,
        ep=30,
        lrr=5e-4
    ):
    """
    Same idea as fine_tune_incremental, but only trains on the k-hop
    induced subgraph around affected nodes. Uses cached AX for those nodes.
    """
    # 1) Build subgraph
    G_sub, sub_nodes = build_k_hop_subgraph(G_updated, affected_nodes, k=k)
    sub_nodes = sorted(sub_nodes)
    idx_map = {n: i for i, n in enumerate(sub_nodes)}

    # 2) Create a PyG data object for the subgraph
    data_sub = from_networkx(G_sub)
    data_sub.x = data.x[[data.node_to_idx[n] for n in sub_nodes]].clone()
    data_sub.y = data.y[[data.node_to_idx[n] for n in sub_nodes]].clone()
    data_sub.nodes_sorted = sub_nodes
    data_sub.node_to_idx = idx_map

    # 3) AX update but only for sub-nodes
    # locally compute adjacency/deg for the subgraph
    adjacency_sub = {n: set(G_sub.neighbors(n)) | {n} for n in sub_nodes}
    deg_sub = {n: len(adjacency_sub[n]) for n in sub_nodes}

    # compute full AX for the subgraph only (small!)
    AX_sub = compute_AX_full(adjacency_sub, deg_sub, sub_nodes, idx_map, data_sub.x)
    data_sub.ax = AX_sub

    # 4) masks: we train on all subgraph nodes
    N = len(sub_nodes)
    mask = torch.ones(N, dtype=torch.bool)
    data_sub.train_mask = mask
    data_sub.val_mask = mask
    data_sub.test_mask = mask

    # 5) fine-tune
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    data_sub = data_sub.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lrr)

    t0 = time.time()
    for epoch in range(1, ep + 1):
        model.train()
        opt.zero_grad()
        out = model(data_sub)
        loss = F.nll_loss(out[mask], data_sub.y[mask])
        loss.backward()
        opt.step()
    t1 = time.time()
    print(f"Subgraph fine-tune time (k={k}): {t1 - t0:.4f}s")

    return model

def compute_weight_svd(model, tag="model"):
    """
    Compute SVD of all Linear layers inside the model.
    Returns dictionary of singular values.
    """
    svd_dict = {}

    for name, param in model.named_parameters():
        if "weight" in name and len(param.shape) == 2:
            W = param.detach().cpu()
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                svd_dict[name] = S.numpy()
                print(f"[{tag}] SVD computed for {name} | shape={W.shape} | top-5 singular values: {S[:5].numpy()}")
            except Exception as e:
                print(f"[{tag}] SVD failed for {name}: {e}")

    return svd_dict

def compare_svd(svd_old, svd_new, tag="comparison"):
    print(f"\n--- Spectral Drift Analysis: {tag} ---")
    for key in svd_old:
        if key in svd_new:
            min_len = min(len(svd_old[key]), len(svd_new[key]))
            diff = np.linalg.norm(svd_old[key][:min_len] - svd_new[key][:min_len])
            print(f"{key} | spectral L2 diff = {diff:.6f}")
# -------------------------
# Timing Benchmark Utilities
# -------------------------

def benchmark_AX_update(adjacency, deg, nodes_sorted, node_to_idx, X, AX, rows_to_update):
    """Benchmark incremental vs full AX recomputation."""
    import time

    # --- Full recompute ---
    start_full = time.time()
    AX_full = compute_AX_full(adjacency, deg, nodes_sorted, node_to_idx, X)
    end_full = time.time()
    full_time = end_full - start_full

    # --- Incremental recompute ---
    start_inc = time.time()
    AX_inc = update_AX_rows(adjacency, deg, nodes_sorted, node_to_idx, X, AX.clone(), rows_to_update)
    end_inc = time.time()
    inc_time = end_inc - start_inc

    # measure error (should be zero on touched rows)
    diff = (AX_full - AX_inc).abs().max().item()

    print("===== AX UPDATE BENCHMARK =====")
    print(f"Full AX recompute time       : {full_time:.4f} s")
    print(f"Incremental AX update time   : {inc_time:.4f} s")
    print(f"Speedup                      : {full_time / inc_time if inc_time>0 else float('inf'):.2f}x")
    print(f"Max |full - inc| difference  : {diff:.6e}")
    print("================================")
def get_acc_and_loss(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data)
        pred = logits.argmax(dim=1)
        acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        loss = F.nll_loss(logits[data.test_mask], data.y[data.test_mask]).item()
    return acc, loss




In [4]:
compare_svd(svd_orig, svd_prev, tag=f"Drift after iteration {count}")


--- Spectral Drift Analysis: Drift after iteration 9 ---
lin1.weight | spectral L2 diff = 0.156738
conv2.lin.weight | spectral L2 diff = 0.046567


In [28]:



import networkx as nx
import numpy as np

def load_pubmed_graph(node_file, cite_file):

    G = nx.Graph()

    node_features = {}
    node_labels = {}
    vocab = set()

    # ----------------------------
    # PASS 1: Collect vocabulary
    # ----------------------------
    with open(node_file, "r") as f:
        for line in f:
            if line.startswith("#") or line.strip()=="":
                continue

            parts = line.strip().split('\t')

            for item in parts[2:]:
                if item.startswith("summary="):
                    continue

                name, value = item.split('=')
                vocab.add(name)

    vocab = sorted(list(vocab))
    vocab_index = {w:i for i,w in enumerate(vocab)}

    # ----------------------------
    # PASS 2: Read nodes
    # ----------------------------
    with open(node_file, "r") as f:

        for line in f:
            if line.startswith("#") or line.strip()=="":
                continue

            parts = line.strip().split('\t')

            node_id = parts[0]

            # label=1 format
            label = int(parts[1].split('=')[1])

            features = np.zeros(len(vocab))

            for item in parts[2:]:

                if item.startswith("summary="):
                    continue

                name, value = item.split('=')

                idx = vocab_index[name]

                features[idx] = float(value)

            node_features[node_id] = features
            node_labels[node_id] = label

            G.add_node(node_id)

    # ----------------------------
    # Read citation edges
    # ----------------------------
    # -----------------------------
    # LOAD EDGES
    # -----------------------------
    with open(cite_file, 'r') as f:

        for line in f:

            parts = line.strip().split('\t')

            if len(parts) < 4:
                continue

            src = parts[1].replace("paper:", "")
            dst = parts[3].replace("paper:", "")

            if src in G and dst in G:
                G.add_edge(src, dst)
    # ----------------------------
    # Attach attributes
    # ----------------------------
    for node in G.nodes():
        G.nodes[node]['x'] = node_features[node]
        G.nodes[node]['y'] = node_labels[node]

    print("Graph loaded")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
    print("Feature dimension:", len(vocab))

    return G

In [5]:

# ---- Split edges into 70% initial + 10 batches ----

def generate_edge_batches(G, initial_ratio=0.7, batches=10):

    edges = list(G.edges())
    random.shuffle(edges)

    n_edges = len(edges)
    initial_edges = int(initial_ratio * n_edges)

    G_initial = nx.Graph()
    G_initial.add_nodes_from(G.nodes(data=True))
    G_initial.add_edges_from(edges[:initial_edges])

    remaining_edges = edges[initial_edges:]
    batch_size = len(remaining_edges) // batches

    edge_batches = []
    for i in range(batches):
        start = i * batch_size
        end = (i + 1) * batch_size
        edge_batches.append(remaining_edges[start:end])

    print("Initial graph edges:", G_initial.number_of_edges())
    print("Edges per batch:", batch_size)

    return G_initial, edge_batches


In [6]:

# ---- Apply batch to graph ----

def apply_edge_batch(G, batch_edges):

    added_edges = []

    for u,v in batch_edges:
        if not G.has_edge(u,v):
            G.add_edge(u,v)
            added_edges.append((u,v))

    affected_nodes = set()
    for u,v in added_edges:
        affected_nodes.add(u)
        affected_nodes.add(v)

    removed_edges = []

    return G, added_edges, removed_edges, affected_nodes


In [44]:
import torch

def compute_important_rows(W, k=5, top_k=5):
    """
    W: torch tensor (m x n)
    Returns indices of top-k important rows
    """
    # SVD
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    k = min(k, U.shape[1])
    # Top-k singular vectors
    U_k = U[:, :k]

    # Leverage scores
    scores = torch.sum(U_k**2, dim=1)
    top_k = min(top_k,scores.shape[0])
    # Top important rows
    important_rows = torch.topk(scores, top_k).indices

    return important_rows

In [39]:
def get_important_rows_all_layers(model, k=5, top_k=5):
    W1 = model.conv1.lin.weight.data
    W2 = model.conv2.lin.weight.data

    imp_rows_W1 = compute_important_rows(W1, k, top_k)
    imp_rows_W2 = compute_important_rows(W2, k, top_k)

    return imp_rows_W1, imp_rows_W2

In [40]:
def mask_gradients(W, important_rows):
    if W.grad is None:
        return

    mask = torch.zeros_like(W.grad)
    mask[important_rows, :] = 1.0
    W.grad *= mask

In [50]:
def fine_tune_selective_rows(model, data, imp_W1, imp_W2, ep=50, lr=5e-4):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(ep):
        optimizer.zero_grad()

        # 🔥 AX cached forward (same as your pipeline)
        out = model(data)

        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])

        loss.backward()

        # 🔥 Apply row masking
        W1 = model.lin1.weight
        W2 = model.conv2.lin.weight

        mask_gradients(W1, imp_W1)
        mask_gradients(W2, imp_W2)

        optimizer.step()

    return model

In [51]:
# ---- Example experiment loop for PubMed ----

node_file = "datasets/Pubmed-Diabetes/data/Pubmed-Diabetes.NODE.paper.tab"
cite_file = "datasets/Pubmed-Diabetes/data/Pubmed-Diabetes.DIRECTED.cites.tab"

G_full = load_pubmed_graph(node_file, cite_file)

# split edges (70% initial + 10 batches)
G_initial, edge_batches = generate_edge_batches(G_full)

print("Total batches:", len(edge_batches))

G = copy.deepcopy(G_initial)

# original labels
node_labels = {n: G_full.nodes[n]['y'] for n in G_full.nodes()}


# ---------------------------------------------------------
# Create initial PyG Data object (SAFE VERSION)
# ---------------------------------------------------------

nodes_sorted = sorted(G_initial.nodes())
node_to_idx = {n:i for i,n in enumerate(nodes_sorted)}

# edge index
edge_index = torch.tensor(
    [[node_to_idx[u], node_to_idx[v]] for u,v in G_initial.edges()],
    dtype=torch.long
).t().contiguous()

data = Data()
data.edge_index = edge_index


# features
data.x = torch.stack([torch.tensor(G_initial.nodes[n]['x']) for n in nodes_sorted]).float()

# ---- Feature normalization ----
data.x = (data.x - data.x.mean(0)) / (data.x.std(0) + 1e-6)
data.x = torch.nan_to_num(data.x)
# labels
data.y = torch.tensor(
    [torch.tensor(G_initial.nodes[n]['y'] - 1) for n in nodes_sorted],
    dtype=torch.long
)

data.nodes_sorted = nodes_sorted
data.node_to_idx = node_to_idx

# build adjacency + AX cache
data = prepare_ax_and_structures(G_initial, data)

# ---------------------------------------------------------
# Create train/val/test masks
# ---------------------------------------------------------

num_nodes = data.num_nodes

perm = torch.randperm(num_nodes)

train_size = int(0.6 * num_nodes)
val_size = int(0.2 * num_nodes)

train_idx = perm[:train_size]
val_idx = perm[train_size:train_size + val_size]
test_idx = perm[train_size + val_size:]

data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

data.train_mask[train_idx] = True
data.val_mask[val_idx] = True
data.test_mask[test_idx] = True


# ---------------------------------------------------------
# Training masks (keep same as original pipeline)
# ---------------------------------------------------------

data.train_mask = data.train_mask
data.val_mask = data.val_mask
data.test_mask = data.test_mask

print("\n====================================")
print("Training INITIAL model on G_initial")
print("====================================")

model_orig = IncrementalGCN(
    in_channels=data.num_features,
    hidden_channels=128,
    out_channels=len(set(node_labels.values()))
)

t_start = time.time()

model_orig = train_model(
    model_orig,
    data,
    e=500,
    lr=0.01
)

t_end = time.time()

initial_train_time = t_end - t_start

print(f"Initial training time: {initial_train_time:.4f} s")

print("\nEvaluation on initial graph:")
evaluate_model(model_orig, data)

acc_init, loss_init = get_acc_and_loss(model_orig, data)

print("\n=== SVD: Initial Model ===")

svd_orig = compute_weight_svd(model_orig, tag="Original")

# create copies used later in experiments
model_orig1 = copy.deepcopy(model_orig)
model_orig2 = copy.deepcopy(model_orig)



# =========================================================
# ITERATION LOOP
# =========================================================

for count, batch in enumerate(edge_batches):

    print("\n==============================")
    print("Iteration:", count)
    print("==============================")

    G_updated, added_edges, removed_edges, affected_nodes = apply_edge_batch(G, batch)
    # ensure node attributes exist
    for n in G_updated.nodes():
        G_updated.nodes[n]['x'] = G_full.nodes[n]['x']
        G_updated.nodes[n]['y'] = G_full.nodes[n]['y']
    nodes_sorted = sorted(G_updated.nodes())
    node_to_idx = {n:i for i,n in enumerate(nodes_sorted)}

    # ---------------------------------------------------------
    # Build updated Data object SAFELY
    # ---------------------------------------------------------

    edge_index = torch.tensor(
        [[node_to_idx[u], node_to_idx[v]] for u,v in G_updated.edges()],
        dtype=torch.long
    ).t().contiguous()

    data_updated = Data()
    data_updated.edge_index = edge_index

    data_updated.x = torch.stack(
        [torch.tensor(G_updated.nodes[n]['x']) for n in nodes_sorted]
    ).float()
    
    # normalize features
    data_updated.x = (data_updated.x - data_updated.x.mean(0)) / (data_updated.x.std(0) + 1e-6)
    data_updated.x = torch.nan_to_num(data_updated.x)
    data_updated.y = torch.tensor(
        [torch.tensor(G_updated.nodes[n]['y'] - 1) for n in nodes_sorted],
        dtype=torch.long
    )
    

    data_updated.nodes_sorted = nodes_sorted
    data_updated.node_to_idx = node_to_idx

    # masks
    data_updated.train_mask = data.train_mask
    data_updated.val_mask = data.val_mask
    data_updated.test_mask = data.test_mask


    # -------------------------------------------------
    # CASE 1: Retrain from scratch
    # -------------------------------------------------

    print('\n=== CASE 1: Retrain from scratch on updated graph ===')

    data_retrain = prepare_ax_and_structures(G_updated, data_updated)
    # restore labels
    data_retrain.y = data_updated.y
    # print(data_updated.y)
    # restore masks
    data_retrain.train_mask = data_updated.train_mask
    data_retrain.val_mask = data_updated.val_mask
    data_retrain.test_mask = data_updated.test_mask
    
    model_retrain = IncrementalGCN(
        in_channels=data_retrain.num_features,
        hidden_channels=128,
        out_channels=int(data_retrain.y.max().item()) + 1
    )

    t_start = time.time()

    model_retrain = train_model(model_retrain, data_retrain, e=500, lr=0.005)

    retrain_time = time.time() - t_start

    print(f"Retrain time: {retrain_time:.4f} s")

    evaluate_model(model_retrain, data_retrain)

    acc_retrain, loss_retrain = get_acc_and_loss(model_retrain, data_retrain)

    print("\n=== SVD: Retrained Model ===")
    svd_retrain = compute_weight_svd(model_retrain, tag="Retrain")

    compare_svd(svd_orig, svd_retrain, tag="Original vs Retrain")


    # -------------------------------------------------
    # CASE 2: Original model (no update)
    # -------------------------------------------------

    print('\n=== CASE 2: Original model applied to updated graph ===')

    t_start = time.time()

    evaluate_model(model_orig, data_updated)

    no_update_time = time.time() - t_start

    print(f"Evaluation time: {no_update_time:.4f} s")

    svd_no_update = compute_weight_svd(model_orig, tag="NoUpdate")

    compare_svd(svd_orig, svd_no_update, tag="Original vs NoUpdate")


    # -------------------------------------------------
    # CASE 3: Cached AX incremental update
    # -------------------------------------------------

    print('\n=== CASE 3: Cached-AX incremental update ===')

    model_cached = copy.deepcopy(model_orig1)

    t_start = time.time()

    model_cached = fine_tune_incremental(
        model_cached,
        G_updated,
        data_updated,
        added_edges,
        removed_edges,
        affected_nodes,
        ep=50,
        lrr=5e-4
    )

    cached_time = time.time() - t_start

    print(f"Cached AX incremental time: {cached_time:.4f} s")

    evaluate_model(model_cached, data_updated)

    svd_cached = compute_weight_svd(model_cached, tag="CachedAX")

    compare_svd(svd_orig, svd_cached, tag="Original vs CachedAX")


    # -------------------------------------------------
    # CASE 4: Subgraph incremental GNN
    # -------------------------------------------------

    print('\n=== CASE 4: Incremental SUBGRAPH GNN (k=2) ===')

    model_sub = copy.deepcopy(model_orig2)

    t_start = time.time()

    model_sub = fine_tune_incremental_subgraph(
        model_sub,
        G_updated,
        data_updated,
        added_edges,
        removed_edges,
        affected_nodes,
        k=2,
        ep=200,
        lrr=5e-4
    )

    subgraph_time = time.time() - t_start

    evaluate_model(model_sub, data_updated)

    svd_sub = compute_weight_svd(model_sub, tag="Subgraph")

    compare_svd(svd_orig, svd_sub, tag="Original vs Subgraph")

    # -------------------------------------------------
    # CASE 5: Cached AX + SVD Row-Selective Update
    # -------------------------------------------------
    
    print('\n=== CASE 5: Cached AX + SVD Row-Selective Update ===')
    
    model_sel = copy.deepcopy(model_orig1)
    
    # Step 1: Prepare AX cache
    data_sel = prepare_ax_and_structures(G_updated, data_updated)
    
    # Step 2: Compute important rows ONLY ONCE
    W1 = model_sel.lin1.weight
    W2 = model_sel.conv2.lin.weight
    
    imp_W1 = compute_important_rows(W1, k=5, top_k=5)
    imp_W2 = compute_important_rows(W2, k=5, top_k=5)
    
    # Step 3: Fine-tune with masked gradients
    t_start = time.time()
    
    model_sel = fine_tune_selective_rows(
        model_sel,
        data_sel,
        imp_W1,
        imp_W2,
        ep=50,
        lr=5e-4
    )
    
    sel_time = time.time() - t_start
    
    print(f"Selective update time: {sel_time:.4f} s")
    
    # Evaluation
    evaluate_model(model_sel, data_sel)
    
    # SVD comparison
    svd_sel = compute_weight_svd(model_sel, tag="Selective")
    
    compare_svd(svd_orig, svd_sel, tag="Original vs Selective")


    # -------------------------------------------------
    # Update iteration state
    # -------------------------------------------------

    data = prepare_ax_and_structures(G_updated, data_updated)
    model_orig1 = copy.deepcopy(model_cached)
    model_orig2 = copy.deepcopy(model_sub)
    svd_prev = compute_weight_svd(model_cached, tag=f"Iteration_{count}_Cached")

    G = copy.deepcopy(G_updated)

Graph loaded
Nodes: 19717
Edges: 44327
Feature dimension: 500
Initial graph edges: 31028
Edges per batch: 1329
Total batches: 10

Training INITIAL model on G_initial
Time taken for training: 35.8557 s
Initial training time: 35.9830 s

Evaluation on initial graph:
Test Accuracy: 0.8387
              precision    recall  f1-score   support

           0     0.8037    0.7948    0.7993       809
           1     0.8526    0.8261    0.8391      1604
           2     0.8428    0.8752    0.8587      1531

    accuracy                         0.8387      3944
   macro avg     0.8331    0.8320    0.8324      3944
weighted avg     0.8388    0.8387    0.8386      3944


=== SVD: Initial Model ===
[Original] SVD computed for lin1.weight | shape=torch.Size([128, 500]) | top-5 singular values: [3.3165185 2.7708175 2.5916054 2.394163  2.2993796]
[Original] SVD computed for conv2.lin.weight | shape=torch.Size([3, 128]) | top-5 singular values: [5.568799  4.9007316 1.0556883]

Iteration: 0

=== CASE 1: